# SVM Dataflow Visualization: From Raw Events to Final Predictions

This notebook is a **visual, step-by-step walkthrough** of the Envision Perdido SVM classifier pipeline for event labeling (`community` vs `non-community`).

It is designed for two goals:

1. **Debugging**: trace how data changes at each stage and inspect failure modes.
2. **Learning**: build intuition for TF-IDF features, SVM margins, support vectors, and kernel behavior.

We will trace this full dataflow:

`raw dataframe -> cleaned/engineered features -> vectorized sparse matrix -> split -> fitted SVM -> decision scores -> predictions -> misclassification analysis`

## 1) Imports and Setup
Uses pandas, numpy, matplotlib, seaborn, scipy sparse tools, and scikit-learn. Optional interactive plotting with plotly is guarded so the notebook still runs without it.

In [ ]:
from __future__ import annotations

import sys
import re
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import sparse

from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_classification

warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

PLOTLY_AVAILABLE = False

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists() and (BASE_DIR.parent / 'data').exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f'Base directory: {BASE_DIR}')
print('Plotly section disabled by default for reproducible runs.')

In [ ]:
# Reuse project pipeline logic where possible
from scripts.ml.svm_train_from_file import build_features, make_series_id

try:
    from scripts.tooling.event_normalizer import clean_text_for_price_detection
except Exception:
    clean_text_for_price_detection = None

print('Imported build_features and make_series_id from scripts.ml.svm_train_from_file')
print(f'clean_text_for_price_detection available: {clean_text_for_price_detection is not None}')

## 2) Reusable Helper Functions
These helpers keep each stage consistent: they print shape/type, preview transformed outputs, and generate common visual diagnostics.

In [ ]:
@dataclass
class StageRecord:
    name: str
    obj_type: str
    shape: tuple | str
    note: str


STAGES: list[StageRecord] = []


def record_stage(name: str, obj, note: str = '') -> None:
    if hasattr(obj, 'shape'):
        shape = obj.shape
    elif isinstance(obj, (list, tuple)):
        shape = (len(obj),)
    else:
        shape = 'n/a'
    rec = StageRecord(name=name, obj_type=type(obj).__name__, shape=shape, note=note)
    STAGES.append(rec)
    print(f'[Stage] {name} | type={rec.obj_type} | shape={rec.shape} | {note}')


def stage_table() -> pd.DataFrame:
    return pd.DataFrame([r.__dict__ for r in STAGES])


def inspect_df(df: pd.DataFrame, name: str = 'DataFrame', head_n: int = 5) -> None:
    print(f'[{name}] shape={df.shape}')
    display(df.head(head_n))
    display(df.dtypes.rename('dtype').to_frame())
    nulls = df.isna().sum().sort_values(ascending=False)
    display(nulls[nulls > 0].rename('null_count').to_frame())


def show_missingness(df: pd.DataFrame, top_n: int = 30) -> None:
    miss = (df.isna().mean() * 100).sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(11, 5))
    sns.barplot(x=miss.values, y=miss.index, orient='h', color='#4C78A8')
    plt.title('Missingness by Column (%)')
    plt.xlabel('Percent Missing')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()


def class_balance_plot(df: pd.DataFrame, label_col: str) -> None:
    if label_col not in df.columns:
        print(f'Column not found: {label_col}')
        return
    vc = df[label_col].value_counts(dropna=False).sort_index()
    plt.figure(figsize=(7, 4))
    sns.barplot(x=vc.index.astype(str), y=vc.values, palette='Set2')
    plt.title('Class Balance')
    plt.xlabel(label_col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()


def sparse_stats(X, name: str = 'Sparse Matrix') -> None:
    if sparse.issparse(X):
        nnz = X.nnz
        total = X.shape[0] * X.shape[1]
        density = nnz / total if total else 0
        print(f'[{name}] shape={X.shape}, nnz={nnz:,}, density={density:.6f}, sparsity={1-density:.6f}')
    else:
        arr = np.asarray(X)
        nnz = np.count_nonzero(arr)
        total = arr.size
        density = nnz / total if total else 0
        print(f'[{name}] dense shape={arr.shape}, density={density:.6f}')


def top_terms_for_doc(vectorizer, sparse_row, top_n: int = 12) -> pd.DataFrame:
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    row = sparse_row.toarray().ravel()
    nz = np.where(row > 0)[0]
    if len(nz) == 0:
        return pd.DataFrame(columns=['term', 'weight'])
    idx = nz[np.argsort(row[nz])[::-1][:top_n]]
    return pd.DataFrame({'term': feature_names[idx], 'weight': row[idx]}).sort_values('weight', ascending=False)


def decision_confidence(scores: np.ndarray) -> np.ndarray:
    # Match project behavior: confidence based on absolute margin from hyperplane
    return 1 / (1 + np.exp(-np.abs(scores)))


def short_text(x: str, n: int = 90) -> str:
    x = str(x)
    return x if len(x) <= n else x[:n-3] + '...'

## 3) Load Data (Real Project Dataset)
The loader searches common project dataset locations and picks the first existing file.

In [ ]:
DATA_CANDIDATES = [
    BASE_DIR / 'data' / 'labeled' / 'consolidated_events_labeled.csv',
    BASE_DIR / 'data' / 'processed' / 'consolidated_training_data.csv',
    BASE_DIR / 'data' / 'labeled' / 'perdido_events_2025_labeled.csv',
    BASE_DIR / 'data' / 'labeled' / 'perdido_events_sept_labeled.csv',
]

DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('No dataset found in expected locations.')

raw_df = pd.read_csv(DATA_PATH)
record_stage('Raw Event DataFrame', raw_df, f'Loaded from {DATA_PATH}')
print(f'Using dataset: {DATA_PATH}')
inspect_df(raw_df, 'Raw Event Data')

In [ ]:
show_missingness(raw_df)

required_defaults = {
    'uid': '',
    'url': '',
    'title': '',
    'description': '',
    'start': '',
    'location': '',
    'label': np.nan,
}

df = raw_df.copy()
for c, default in required_defaults.items():
    if c not in df.columns:
        df[c] = default

record_stage('Schema-normalized DataFrame', df, 'Added any missing expected columns with safe defaults')
print(f'Duplicate rows: {df.duplicated().sum():,}')

## 4) Raw Data Inspection
Inspect class balance, sample rows from both classes, text lengths, source distribution, and duplicates.

In [ ]:
label_series = df['label'].astype(str).str.replace('.0', '', regex=False).str.strip()
df['label_clean'] = pd.to_numeric(label_series.where(label_series.isin(['0', '1'])), errors='coerce')

class_balance_plot(df.dropna(subset=['label_clean']), 'label_clean')

if 'source' in df.columns:
    src = df['source'].fillna('missing').astype(str).value_counts().head(15)
    plt.figure(figsize=(9, 5))
    sns.barplot(x=src.values, y=src.index, color='#72B7B2')
    plt.title('Top Sources')
    plt.xlabel('Count')
    plt.ylabel('source')
    plt.tight_layout()
    plt.show()

for col in ['title', 'description']:
    lens = df[col].fillna('').astype(str).str.len()
    plt.figure(figsize=(8, 4))
    sns.histplot(lens, bins=40, kde=True, color='#F58518')
    plt.title(f'Document Length Distribution: {col}')
    plt.xlabel('Characters')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

display(df[df['label_clean'] == 1][['title', 'description', 'location', 'label_clean']].head(5))
display(df[df['label_clean'] == 0][['title', 'description', 'location', 'label_clean']].head(5))

## 5) Step-by-Step Preprocessing Visualization
Each transformation below reports before/after state so you can track how raw rows become model-ready features.

In [ ]:
before_n = len(df)
df_prep = df.copy()

# Keep only rows with valid binary labels
mask = df_prep['label_clean'].isin([0, 1])
df_prep = df_prep.loc[mask].copy()
df_prep['label_clean'] = df_prep['label_clean'].astype(int)

for col in ['title', 'description', 'location', 'start', 'uid', 'url']:
    df_prep[col] = df_prep[col].fillna('').astype(str)

if clean_text_for_price_detection is not None:
    df_prep['title_clean'] = df_prep['title'].map(clean_text_for_price_detection)
    df_prep['description_clean'] = df_prep['description'].map(clean_text_for_price_detection)
else:
    cleaner = lambda x: re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', str(x))).strip()
    df_prep['title_clean'] = df_prep['title'].map(cleaner)
    df_prep['description_clean'] = df_prep['description'].map(cleaner)

df_prep['combined_text'] = (df_prep['title_clean'] + ' ' + df_prep['description_clean']).str.strip()

if 'series_id' not in df_prep.columns:
    df_prep['series_id'] = make_series_id(df_prep, 'uid', 'url', 'title_clean', 'location')

record_stage('Labeled+cleaned DataFrame', df_prep, f'Filtered {before_n} -> {len(df_prep)} rows with labels 0/1')
display(df_prep[['title', 'title_clean', 'description', 'description_clean', 'combined_text']].head(3))

In [ ]:
# Build project-aligned structured+text model input frame
tmp_for_features = df_prep.copy()
tmp_for_features['title'] = tmp_for_features['title_clean']
tmp_for_features['description'] = tmp_for_features['description_clean']
X_df, num_cols = build_features(
    tmp_for_features,
    title_col='title',
    desc_col='description',
    start_col='start',
    loc_col='location',
)

y = df_prep['label_clean'].values
record_stage('Feature Frame (pre-vectorization)', X_df, 'Output of build_features()')
print(f'Numeric feature columns: {num_cols}')
display(X_df.head())

## 6) Load Trained Pipeline and Inspect TF-IDF Feature Space
This section shows what TF-IDF actually produces: vocabulary size, sparse matrix shape, non-zero patterns, and strongest terms for individual events.

In [ ]:
MODEL_CANDIDATES = [
    BASE_DIR / 'models' / 'community_svm.pkl',
    BASE_DIR / 'data' / 'artifacts' / 'event_classifier_model.pkl',
]

model_path = next((p for p in MODEL_CANDIDATES if p.exists()), None)
if model_path is None:
    raise FileNotFoundError('No model artifact found in expected locations.')

import joblib
model_obj = joblib.load(model_path)

if isinstance(model_obj, dict) and 'pipe' in model_obj:
    pipe = model_obj['pipe']
elif isinstance(model_obj, Pipeline):
    pipe = model_obj
else:
    raise ValueError(f'Unsupported model format at {model_path}')

pre = pipe.named_steps.get('pre')
clf = pipe.named_steps.get('clf')
if pre is None or clf is None:
    raise ValueError('Expected sklearn Pipeline with named steps pre and clf.')

record_stage('Loaded trained pipeline', pipe, f'Model path: {model_path.name}')
print(f'Model path: {model_path}')
print(pipe)

In [ ]:
X_trans = pre.transform(X_df)
record_stage('Vectorized feature matrix', X_trans, 'Output of pre.transform(X_df)')
sparse_stats(X_trans, 'TF-IDF + Numeric Feature Space')

feature_names = pre.get_feature_names_out()
print(f'Feature count (vocabulary + numeric): {len(feature_names):,}')

nonzero_per_row = np.diff(X_trans.tocsr().indptr) if sparse.issparse(X_trans) else np.count_nonzero(X_trans, axis=1)
plt.figure(figsize=(9, 4))
sns.histplot(nonzero_per_row, bins=40, kde=True, color='#54A24B')
plt.title('Non-zero Features per Event Row')
plt.xlabel('Non-zero feature count')
plt.ylabel('Rows')
plt.tight_layout()
plt.show()

In [ ]:
# Pull TF-IDF vectorizer and inspect vocabulary-level behavior
txt_vec = pre.named_transformers_['txt'] if hasattr(pre, 'named_transformers_') and 'txt' in pre.named_transformers_ else None

if txt_vec is None:
    print('Could not access text vectorizer from preprocessor.')
else:
    vocab_size = len(txt_vec.vocabulary_)
    print(f'TF-IDF vocabulary size: {vocab_size:,}')

    X_txt = txt_vec.transform(X_df['text'])
    sparse_stats(X_txt, 'Text-only TF-IDF Matrix')

    df_counts = (X_txt > 0).sum(axis=0).A1
    terms = np.asarray(txt_vec.get_feature_names_out())
    top_idx = np.argsort(df_counts)[::-1][:25]
    top_df = pd.DataFrame({'term': terms[top_idx], 'doc_frequency': df_counts[top_idx]})

    plt.figure(figsize=(10, 7))
    sns.barplot(data=top_df.sort_values('doc_frequency', ascending=True), x='doc_frequency', y='term', color='#4C78A8')
    plt.title('Most Common Vocabulary Terms by Document Frequency')
    plt.xlabel('Document Frequency')
    plt.ylabel('Term')
    plt.tight_layout()
    plt.show()

    sample_i = min(10, len(df_prep) - 1)
    doc_top = top_terms_for_doc(txt_vec, X_txt[sample_i], top_n=15)
    display(pd.DataFrame({
        'original_text': [short_text(X_df.iloc[sample_i]['text'], 220)],
        'top_terms': [', '.join(doc_top['term'].head(8).tolist())]
    }))

    plt.figure(figsize=(8, 4))
    sns.barplot(data=doc_top, x='weight', y='term', color='#F58518')
    plt.title('Top TF-IDF Weights for One Example Event')
    plt.xlabel('TF-IDF Weight')
    plt.ylabel('Term')
    plt.tight_layout()
    plt.show()

## 7) Pipeline Dataflow Trace
A compact trace table of every major object flowing through the pipeline.

In [ ]:
# Group-aware split if possible (same approach as training script)
groups = df_prep['series_id'].astype(str).fillna('')
if groups.nunique() >= 3 and pd.Series(y).nunique() >= 2 and len(df_prep) >= 20:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X_df, y, groups=groups))
    split_mode = 'group_shuffle_split'
else:
    train_idx, test_idx = train_test_split(np.arange(len(X_df)), test_size=0.2, random_state=42, stratify=y if pd.Series(y).nunique() > 1 else None)
    split_mode = 'stratified_or_random_split'

X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

record_stage('Train split frame', X_train, split_mode)
record_stage('Test split frame', X_test, split_mode)

y_pred = pipe.predict(X_test)
scores = np.asarray(pipe.decision_function(X_test)).ravel()
conf = decision_confidence(scores)

record_stage('Decision scores', scores, 'Raw signed margin distance')
record_stage('Predictions', y_pred, '0=non-community, 1=community')

display(stage_table())
print('Classification Report')
print(classification_report(y_test, y_pred, digits=3))

## 8) Dimensionality Reduction Visualization
Project high-dimensional sparse features into lower-dimensional spaces and compare true labels, predicted labels, and misclassified points.

In [ ]:
X_test_trans = pre.transform(X_test)
svd = TruncatedSVD(n_components=3, random_state=42)
embed3 = svd.fit_transform(X_test_trans)

viz_df = pd.DataFrame({
    'svd1': embed3[:, 0],
    'svd2': embed3[:, 1],
    'svd3': embed3[:, 2],
    'true_label': y_test.astype(int),
    'pred_label': y_pred.astype(int),
    'decision_score': scores,
    'confidence': conf,
})
viz_df['misclassified'] = viz_df['true_label'] != viz_df['pred_label']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=viz_df, x='svd1', y='svd2', hue='true_label', style='misclassified', size='confidence', sizes=(20, 180), alpha=0.8, ax=axes[0])
axes[0].set_title('Projected Event Embeddings (Colored by True Label)')

sns.scatterplot(data=viz_df, x='svd1', y='svd2', hue='pred_label', style='misclassified', size='confidence', sizes=(20, 180), alpha=0.8, ax=axes[1])
axes[1].set_title('Projected Event Embeddings (Colored by Predicted Label)')

for ax in axes:
    ax.axhline(0, color='grey', lw=0.5)
    ax.axvline(0, color='grey', lw=0.5)
plt.tight_layout()
plt.show()

# 3D projection view
fig = plt.figure(figsize=(10, 7))
ax3d = fig.add_subplot(111, projection='3d')
scatter = ax3d.scatter(viz_df['svd1'], viz_df['svd2'], viz_df['svd3'], c=viz_df['true_label'], cmap='Set1', s=40 + 120 * viz_df['misclassified'].astype(int), alpha=0.75)
ax3d.set_title('3D Projected Event Embeddings (True Labels)')
ax3d.set_xlabel('SVD 1')
ax3d.set_ylabel('SVD 2')
ax3d.set_zlabel('SVD 3')
plt.tight_layout()
plt.show()

# annotate a few points
annot_idx = viz_df.sample(min(8, len(viz_df)), random_state=42).index
x_test_reset = X_test.reset_index(drop=True)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=viz_df, x='svd1', y='svd2', hue='true_label', style='misclassified', alpha=0.7)
for i in annot_idx:
    title = short_text(x_test_reset.iloc[i]['text'] if 'text' in x_test_reset.columns else '', 35)
    plt.annotate(title, (viz_df.loc[i, 'svd1'], viz_df.loc[i, 'svd2']), fontsize=8, alpha=0.75)
plt.title('Projected Embeddings with Example Event Annotations')
plt.tight_layout()
plt.show()

In [ ]:
# Optional t-SNE for smaller subsets (can be slow)
MAX_TSNE = 800
if len(X_test) <= MAX_TSNE:
    dense_small = X_test_trans.toarray() if sparse.issparse(X_test_trans) else np.asarray(X_test_trans)
    tsne = TSNE(n_components=2, random_state=42, init='pca', learning_rate='auto', perplexity=min(30, max(5, len(X_test)//8)))
    ts = tsne.fit_transform(dense_small)
    tsne_df = pd.DataFrame({'x': ts[:, 0], 'y': ts[:, 1], 'true_label': y_test, 'pred_label': y_pred})
    tsne_df['misclassified'] = tsne_df['true_label'] != tsne_df['pred_label']

    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=tsne_df, x='x', y='y', hue='true_label', style='misclassified', alpha=0.8)
    plt.title('t-SNE Projection (Optional)')
    plt.tight_layout()
    plt.show()
else:
    print(f'Skipping t-SNE because test size ({len(X_test)}) > {MAX_TSNE}')

## 9) SVM Geometry Visualization (Educational Section)
Visualize a linear SVM in 2D with decision boundary, margins, and support vectors.

In [ ]:
# Toy 2D data for explicit geometry visualization
X_toy, y_toy = make_classification(
    n_samples=180, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.3, random_state=42
)

lin = SVC(kernel='linear', C=1.0)
lin.fit(X_toy, y_toy)

def plot_linear_svm_geometry(model, X, y, title='Support Vectors and Margin'):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette='Set1', alpha=0.8, edgecolor='black', linewidth=0.3)

    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    xx = np.linspace(xlim[0], xlim[1], 250)
    yy = np.linspace(ylim[0], ylim[1], 250)
    YY, XX = np.meshgrid(yy, xx)
    grid = np.vstack([XX.ravel(), YY.ravel()]).T
    Z = model.decision_function(grid).reshape(XX.shape)

    plt.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.8, linestyles=['--', '-', '--'])
    sv = model.support_vectors_
    plt.scatter(sv[:, 0], sv[:, 1], s=180, facecolors='none', edgecolors='black', linewidth=1.8, label='Support vectors')

    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_linear_svm_geometry(lin, X_toy, y_toy)
print('Support vectors are the points that define the margin; moving them changes the boundary most.')

## 10) Kernel Trick Demonstration
Compare linear vs RBF on nonlinear toy data, then bridge the intuition back to real text/event features.

In [ ]:
X_moons, y_moons = make_moons(n_samples=240, noise=0.2, random_state=42)

lin_m = SVC(kernel='linear', C=1.0)
rbf_m = SVC(kernel='rbf', C=1.5, gamma='scale')
lin_m.fit(X_moons, y_moons)
rbf_m.fit(X_moons, y_moons)

def plot_boundary(ax, model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.8, X[:, 0].max() + 0.8
    y_min, y_max = X[:, 1].min() - 0.8, X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    z = model.decision_function(grid).reshape(xx.shape)

    ax.contourf(xx, yy, z, levels=30, cmap='coolwarm', alpha=0.25)
    ax.contour(xx, yy, z, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='Set1', s=35, edgecolor='black', linewidth=0.2)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_boundary(axes[0], lin_m, X_moons, y_moons, 'Linear SVM on Nonlinear Data (Struggles)')
plot_boundary(axes[1], rbf_m, X_moons, y_moons, 'RBF SVM on Nonlinear Data (Succeeds)')
plt.tight_layout()
plt.show()

### Kernel Intuition (Plain-English)
A kernel lets SVM behave **as if** data were mapped into a richer feature space where separation is easier, without explicitly constructing every high-dimensional coordinate.

In this project, TF-IDF text vectors already live in a high-dimensional sparse space. Kernel methods can still help when class boundaries are nonlinear in that space.

## 11) Real Model Decision Analysis
Inspect decision-function distributions, near-boundary events, and confidence extremes to diagnose class skew.

In [ ]:
eval_df = X_test.copy()
eval_df['true_label'] = y_test
eval_df['pred_label'] = y_pred
eval_df['decision_score'] = scores
eval_df['confidence'] = conf
eval_df['abs_margin'] = np.abs(scores)

plt.figure(figsize=(10, 5))
sns.histplot(data=eval_df, x='decision_score', hue='true_label', bins=40, kde=True, element='step', stat='count')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Decision Function Distribution by True Class')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(data=eval_df, x='decision_score', hue='pred_label', bins=40, kde=True, element='step', stat='count')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Decision Function Distribution by Predicted Class')
plt.tight_layout()
plt.show()

near_boundary = eval_df.nsmallest(20, 'abs_margin')
most_confident = eval_df.nlargest(20, 'confidence')
least_confident = eval_df.nsmallest(20, 'confidence')

print('Events nearest the hyperplane (most ambiguous):')
display(near_boundary[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']])

print('Most confident predictions:')
display(most_confident[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']].head(10))

print('Least confident predictions:')
display(least_confident[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']].head(10))

## 12) Misclassification Analysis
False positives/negatives, confusion matrix, near-boundary errors, and confidently wrong predictions.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['non-community', 'community'], yticklabels=['non-community', 'community'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

mis = eval_df[eval_df['true_label'] != eval_df['pred_label']].copy()
fp = mis[(mis['true_label'] == 0) & (mis['pred_label'] == 1)].copy()
fn = mis[(mis['true_label'] == 1) & (mis['pred_label'] == 0)].copy()

print(f'False positives: {len(fp)} | False negatives: {len(fn)}')

display(fp[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']].sort_values('confidence', ascending=False).head(20))
display(fn[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']].sort_values('confidence', ascending=False).head(20))

print('Nearest-to-boundary errors:')
display(mis.assign(abs_margin=np.abs(mis['decision_score'])).sort_values('abs_margin').head(20)[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']])

print('Most confident wrong predictions:')
display(mis.sort_values('confidence', ascending=False).head(20)[['text', 'true_label', 'pred_label', 'decision_score', 'confidence']])

## 13) Feature Importance and Interpretability
For linear SVM, map coefficients back to feature names and visualize strongest community vs non-community features.

In [ ]:
if hasattr(clf, 'coef_'):
    names = pre.get_feature_names_out()
    coefs = clf.coef_.ravel()
    n = min(len(names), len(coefs))
    coef_df = pd.DataFrame({'feature': names[:n], 'weight': coefs[:n]})

    top_pos = coef_df.nlargest(20, 'weight').copy()
    top_neg = coef_df.nsmallest(20, 'weight').copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    sns.barplot(data=top_pos.sort_values('weight'), x='weight', y='feature', color='#4C78A8', ax=axes[0])
    axes[0].set_title('Top Positive Features (community)')

    sns.barplot(data=top_neg.sort_values('weight', ascending=False), x='weight', y='feature', color='#E45756', ax=axes[1])
    axes[1].set_title('Top Negative Features (non-community)')

    plt.tight_layout()
    plt.show()

    display(top_pos.head(15))
    display(top_neg.head(15))
else:
    print('Classifier does not expose linear coefficients (nonlinear kernel or unsupported model).')
    print('Fallback: rely on local TF-IDF term tables + misclassification slices above for interpretability.')

## 14) Data Drift and Health Clues
Optional checks for imbalance, source skew, repeated patterns in false positives, and vocabulary drift hints.

In [ ]:
# Class imbalance
class_ratio = pd.Series(y).value_counts(normalize=True).sort_index()
print('Class ratio (0=non-community, 1=community):')
display(class_ratio.rename('ratio').to_frame())

# Source imbalance if available
if 'source' in df_prep.columns:
    src_by_class = pd.crosstab(df_prep['source'].fillna('missing'), df_prep['label_clean'])
    display(src_by_class.sort_values(by=src_by_class.columns.tolist(), ascending=False).head(20))

# Repeated false-positive lexical patterns
if len(fp) > 0:
    fp_terms = pd.Series(' '.join(fp['text'].fillna('').str.lower()).split())
    stop = set(['the', 'and', 'for', 'with', 'from', 'your', 'this', 'that', 'you', 'our', 'are', 'at', 'in', 'on', 'to', 'of', 'a'])
    fp_terms = fp_terms[~fp_terms.isin(stop)]
    top_fp_terms = fp_terms.value_counts().head(25)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_fp_terms.values, y=top_fp_terms.index, color='#B279A2')
    plt.title('Frequent Tokens in False Positives (Health Clue)')
    plt.xlabel('Count')
    plt.ylabel('Token')
    plt.tight_layout()
    plt.show()
else:
    print('No false positives in this split.')

## 15) Reusable Helper Toolkit Recap
The notebook uses reusable helpers for future retraining/debugging runs:

- `record_stage`, `stage_table`
- `inspect_df`, `show_missingness`, `class_balance_plot`
- `sparse_stats`, `top_terms_for_doc`, `decision_confidence`

You can rerun this notebook against a new dataset/model and keep the same instrumentation.

## 16) Final Summary
This notebook visualized the complete event classification dataflow: raw rows, cleaning, feature engineering, sparse TF-IDF representation, low-dimensional structure, SVM decision scores, and error analysis.

Use these signals to debug class skew:

1. **Score distributions**: if both classes sit mostly on one side of 0, boundary is biased.
2. **Near-boundary errors**: indicate ambiguous language and weak separation.
3. **Confidently wrong cases**: indicate systematic feature mismatch or label noise.
4. **Top coefficients/tokens**: reveal model shortcuts (possibly overused community cues).

Suggested next debugging steps:

- Review confidently wrong false positives and add targeted negative examples.
- Check source-specific bias and rebalance if one source dominates.
- Revisit threshold/review margin policy after inspecting score overlap.
- Add domain-specific phrase normalization for recurring ambiguous terms.

### TODO (only if needed)
If model artifacts in your environment are legacy/non-pipeline format, adapt the model-loading cell to attach the separate vectorizer and classifier objects before rerunning diagnostics.